# 02 · Data Cleaning

Turn the 60 raw files into **one table**: one row per match, with results, referee decisions, and attendance.

**Steps**
1. Load both sources
2. Remove rows that aren't real league matches
3. Map team names so the two sources agree
4. Merge
5. Classify every match: fans, partial crowd, ghost game, or unknown
6. Build the features for EDA
7. Save `data/clean/matches.csv`

Every step ends with **checks**: ✅ means the data is what we expect, ❌ means something needs a look.

In [1]:
from io import StringIO
from pathlib import Path

import numpy as np
import pandas as pd

RAW = Path("../data/raw")
CLEAN = Path("../data/clean")
CLEAN.mkdir(parents=True, exist_ok=True)

LEAGUES = ["EPL", "La_Liga", "Serie_A", "Bundesliga", "Ligue_1"]
SEASONS = range(2017, 2026)   # 2017-18 ... 2025-26

def check(ok, msg):
    """Print a pass/fail line. Fails don't stop the notebook, they flag something to look at."""
    print(("✅ " if ok else "❌ ") + msg)

## 1. Load both sources

Each file becomes a table with `league` and `season` added, then all 30 are stacked into one.

In [2]:
def load_datahub(league, season):
    df = pd.read_csv(RAW / f"fd_{league}_{season}.csv")
    return df.assign(league=league, season=season)

def load_fbref(league, season):
    html = (RAW / f"fbref_{league}_{season}.html").read_text(encoding="utf-8")
    for t in pd.read_html(StringIO(html)):
        if {"Home", "Away", "Attendance"} <= set(t.columns):
            return t.assign(league=league, season=season)
    raise ValueError(f"No fixtures table in fbref_{league}_{season}.html")

fd = pd.concat([load_datahub(l, s) for l in LEAGUES for s in SEASONS], ignore_index=True)
fb = pd.concat([load_fbref(l, s)   for l in LEAGUES for s in SEASONS], ignore_index=True)
print("DataHub:", fd.shape, "  FBref:", fb.shape)

DataHub: (16111, 24)   FBref: (18097, 15)


**Expected DataHub size:** 380 matches per season for EPL, La Liga and Serie A (20 teams), and 306 for the Bundesliga (18 teams). Ligue 1 had 20 teams (380 matches) until 2022–23 and 18 teams (306) from 2023–24. The exception is Ligue 1 2019–20, which was cancelled in March 2020 after 279 matches.

In [3]:
def expected_matches(league, season):
    if league == "Ligue_1" and season == 2019:
        return 279
    if league == "Bundesliga" or (league == "Ligue_1" and season >= 2023):
        return 306
    return 380

counts = fd.groupby(["league", "season"]).size()
wrong = [(lg, yr, n) for (lg, yr), n in counts.items() if n != expected_matches(lg, yr)]
check(not wrong, f"DataHub has the expected number of matches in all {len(counts)} seasons ({len(fd)} total)")
wrong

✅ DataHub has the expected number of matches in all 45 seasons (16111 total)


[]

## 2. Remove rows that aren't real league matches

FBref has three kinds of extra rows:
- **Blank separator rows** between matchweeks
- **Cancelled matches:** Ligue 1 2019–20 lists the full fixture list, but the season stopped in March 2020
- **Relegation play-offs:** extra matches between a top-flight and a second-division team, which DataHub doesn't include

In [4]:
n0 = len(fb)
fb = fb.dropna(subset=["Home", "Away"])
n1 = len(fb)
cancelled = fb["Score"].isna()
print("cancelled:", fb[cancelled].groupby(["league", "season"]).size().to_dict())
fb = fb[~cancelled]
n2 = len(fb)
playoff = fb["Round"].fillna("").str.contains("play-off|tie-breaker", case=False)
print("play-off rows:", fb[playoff]["Round"].value_counts().to_dict())
fb = fb[~playoff]

print(f"\n{n0} rows -> {n1} after separators -> {n2} after cancelled -> {len(fb)} after play-offs")
check(len(fb) == len(fd), f"FBref now has the same number of matches as DataHub ({len(fb)} vs {len(fd)})")

cancelled: {('Ligue_1', 2019): 101}
play-off rows: {'German 1/2 Relegation/Promotion play-offs': 18, 'French 1/2 Relegation/Promotion play-offs': 14, 'Relegation tie-breaker': 1}

18097 rows -> 16245 after separators -> 16144 after cancelled -> 16111 after play-offs
✅ FBref now has the same number of matches as DataHub (16111 vs 16111)


## 3. Map team names

The two sites spell teams differently (`Man United` vs `Manchester Utd`, `Ath Madrid` vs `Atlético Madrid`). This mapping converts DataHub names to FBref names.

How it was built: matches were paired across the two sources by league, date and score, and the team names in each pair were compared. Only names that differ are listed. Every other team is spelled the same on both sites.

In [5]:
TEAM_MAP = {
    "EPL": {
        "Cardiff": "Cardiff City", "Leeds": "Leeds United", "Leicester": "Leicester City",
        "Man City": "Manchester City", "Man United": "Manchester Utd", "Norwich": "Norwich City",
        "Nott'm Forest": "Nottingham", "Stoke": "Stoke City", "Swansea": "Swansea City",
        "Luton": "Luton Town", "Ipswich": "Ipswich Town",
    },
    "La_Liga": {
        "Alaves": "Alavés", "Almeria": "Almería", "Ath Bilbao": "Athletic Club",
        "Ath Madrid": "Atlético Madrid", "Betis": "Real Betis", "Cadiz": "Cádiz",
        "Celta": "Celta Vigo", "Espanol": "Espanyol", "La Coruna": "Dep. La Coruña",
        "Leganes": "Leganés", "Malaga": "Málaga", "Sociedad": "Real Sociedad",
        "Vallecano": "Rayo Vallecano",
    },
    "Serie_A": {"Spal": "SPAL", "Verona": "Hellas Verona"},
    "Bundesliga": {
        "Bielefeld": "Arminia", "Ein Frankfurt": "Frankfurt", "FC Koln": "Köln",
        "Fortuna Dusseldorf": "Düsseldorf", "Greuther Furth": "Greuther Fürth",
        "Hamburg": "Hamburger SV", "Hannover": "Hannover 96", "Hertha": "Hertha BSC",
        "M'gladbach": "Gladbach", "Mainz": "Mainz 05", "Nurnberg": "Nürnberg",
        "Paderborn": "Paderborn 07", "Darmstadt": "Darmstadt 98",
    },
    "Ligue_1": {"Clermont": "Clermont Foot", "Nimes": "Nîmes", "St Etienne": "Saint-Étienne"},
}

def to_fbref_name(league, team):
    return TEAM_MAP[league].get(team, team)

fd["Home"] = [to_fbref_name(l, t) for l, t in zip(fd["league"], fd["HomeTeam"])]
fd["Away"] = [to_fbref_name(l, t) for l, t in zip(fd["league"], fd["AwayTeam"])]

In [6]:
# every DataHub team should now exist in FBref for the same league
fd_teams = set(zip(fd["league"], fd["Home"]))
fb_teams = set(zip(fb["league"], fb["Home"]))
unmatched = sorted(fd_teams - fb_teams)
check(not unmatched, f"all {len(fd_teams)} teams have a matching FBref name" +
      (f" — unmatched: {unmatched}" if unmatched else ""))

# no two different teams should end up with the same name
key = ["league", "season", "Home", "Away"]
check(not fd.duplicated(key).any(), "no duplicate matches in DataHub after renaming")
check(not fb.duplicated(key).any(), "no duplicate matches in FBref")

✅ all 153 teams have a matching FBref name
✅ no duplicate matches in DataHub after renaming
✅ no duplicate matches in FBref


## 4. Merge

Matches are joined on **league + season + home team + away team**. Each pair of teams meets once at each ground per season, so this identifies every match uniquely. `indicator=True` marks which rows found a partner.

In [7]:
fb_cols = ["Date", "Score", "Attendance", "Venue", "Referee", "Notes"]
df = fd.merge(fb[key + fb_cols], on=key, how="outer",
              suffixes=("_dh", "_fb"), indicator=True)
print(df["_merge"].value_counts())

_merge
both          16111
left_only         0
right_only        0
Name: count, dtype: int64


In [8]:
check((df["_merge"] == "both").all(), f"every match found its partner ({(df['_merge'] == 'both').sum()} matched)")
df = df[df["_merge"] == "both"].drop(columns="_merge")

check((df["Date_dh"] == df["Date_fb"]).all(), "dates agree between sources")

# does the score agree?
fb_score = df["Score"].str.extract(r"(\d+)\D+(\d+)").astype(float)
score_diff = (fb_score[0] != df["FTHG"]) | (fb_score[1] != df["FTAG"])
awarded = df["Notes"].fillna("").str.contains("awarded")
check(not (score_diff & ~awarded).any(),
      f"scores agree between sources, except {score_diff.sum()} match(es) officially awarded afterwards")
df.loc[score_diff, ["league", "season", "Home", "Away", "FTHG", "FTAG", "Score", "Notes"]]

✅ every match found its partner (16111 matched)
✅ dates agree between sources
✅ scores agree between sources, except 1 match(es) officially awarded afterwards


,league,season,Home,Away,FTHG,FTAG,Score,Notes
13977,Serie_A,2020,Hellas Verona,Roma,0,0,3–0,Match awarded to Hellas Verona


**The score difference explained:** Hellas Verona vs Roma (Sep 2020) ended 0–0, but Roma fielded an ineligible player, so the result was officially changed to 3–0 for Verona. DataHub keeps the **on-pitch** result and FBref the **official** one.

We use the on-pitch result, since we're studying what happened on the pitch. Both "awarded" matches stay in the data with a flag, so EDA can exclude them if needed.

In [9]:
df["awarded"] = awarded
df[df["awarded"]][["league", "season", "Home", "Away", "FTHG", "FTAG", "Notes"]]

,league,season,Home,Away,FTHG,FTAG,Notes
1281,Bundesliga,2021,Bochum,Gladbach,0,2,Match awarded to Gladbach
2399,Bundesliga,2024,Union Berlin,Bochum,0,2,Match awarded to Bochum
12246,Ligue_1,2024,Montpellier,Saint-Étienne,0,2,Match awarded to Saint-Étienne
13977,Serie_A,2020,Hellas Verona,Roma,0,0,Match awarded to Hellas Verona
15453,Serie_A,2024,Fiorentina,Inter,3,0,Match awarded to Fiorentina


### Matches that weren't played to the end

Two matches have no usable on-pitch data:
- **Union Berlin vs Bochum (Dec 2024):** a lighter thrown from the stands hit Bochum's goalkeeper. The match was later **awarded 0–2** to Bochum, and DataHub has no match stats for it.
- **Nantes vs Toulouse (May 2026):** the match was **suspended** and never finished.

We remove both, since they don't show what happened in a full match.

In [10]:
no_stats = df[["HS", "AS", "HF", "AF", "HY", "AY"]].isna().all(axis=1)
suspended = df["Notes"].fillna("").str.contains("Suspended")
print(df.loc[no_stats | suspended, ["league", "Date_dh", "Home", "Away", "FTHG", "FTAG", "Notes"]].to_string())

n_before = len(df)
df = df[~(no_stats | suspended)]
check(len(df) == n_before - 2, f"removed 2 incomplete matches ({len(df)} left)")

           league     Date_dh          Home      Away  FTHG  FTAG                    Notes
2399   Bundesliga  2024-12-14  Union Berlin    Bochum     0     2  Match awarded to Bochum
12588     Ligue_1  2026-05-17        Nantes  Toulouse     0     0          Match Suspended
✅ removed 2 incomplete matches (16109 left)


### Which referee column?

DataHub only has referee names for the **EPL**. FBref has them for **every league**, so we use FBref's.

In [11]:
print("missing referee by league (DataHub):")
print(df.groupby("league")["Referee_dh"].apply(lambda s: s.isna().sum()).to_dict())
check(df["Referee_fb"].notna().all(), "FBref has a referee for every match")
df["referee"] = df["Referee_fb"]

missing referee by league (DataHub):
{'Bundesliga': 2753, 'EPL': 0, 'La_Liga': 3420, 'Ligue_1': 3096, 'Serie_A': 3420}
✅ FBref has a referee for every match


## 5. Classify the crowd

Every match gets a `crowd_status`:

| Status | Meaning |
|---|---|
| `fans` | normal crowd |
| `partial` | COVID-era limited crowd (under 50% of normal capacity) |
| `none` | ghost game, no fans |
| `unknown` | attendance missing for another reason, so it's excluded from crowd analysis |

**The tricky part:** FBref leaves attendance **blank** for ghost games, never 0, and it's occasionally blank for normal matches too, when the number just wasn't recorded. So a blank only counts as a ghost game inside the COVID window.

We use **two windows**, based on the table below:
- **Ghost window, 1 Mar 2020 to 31 Jul 2021:** a blank attendance here means no fans.
- **Restriction window, 1 Mar 2020 to 31 Mar 2022:** crowds here were often capped (e.g. 5,000 in Serie A and Ligue 1 in January 2022), so a small crowd counts as `partial`.

In [12]:
df["date"] = pd.to_datetime(df["Date_dh"])
month = df["date"].dt.to_period("M")
window = (df["date"] >= "2020-03-01") & (df["date"] <= "2022-06-30")

# share of matches WITH attendance, by month and league
coverage = (df[window].groupby([month[window], "league"])["Attendance"]
            .apply(lambda s: f"{s.notna().sum()}/{len(s)}").unstack())
coverage

league,Bundesliga,EPL,La_Liga,Ligue_1,Serie_A
date,,,,,
2020-03,11/12,12/12,15/16,12/12,2/8
2020-05,0/35,NaN,NaN,NaN,NaN
2020-06,0/47,0/26,0/53,NaN,0/26
2020-07,NaN,0/66,0/57,NaN,0/88
2020-08,NaN,NaN,NaN,2/16,0/10
2020-09,13/18,0/28,0/30,1/34,16/20
2020-10,19/33,0/34,0/38,1/31,27/31
2020-11,0/30,0/36,0/35,0/37,0/38
2020-12,0/36,12/57,0/52,0/50,0/49


**Reading the table:**
- **March 2020:** most matches still had fans, but Serie A played 6 of 8 behind closed doors before the lockdown.
- **May 2020 to April 2021:** almost all empty. The exceptions are partial crowds in Sep–Oct 2020 (Bundesliga, Serie A), a few EPL matches in Dec 2020, and May 2021, when fans started coming back.
- **From August 2021:** fans are back, but there are scattered blanks, e.g. the **Bundesliga Dec 2021–Jan 2022** (real ghost games during the Omicron wave) and a few in Ligue 1.

Blanks outside the ghost window, like those in 2021–22, are labeled `unknown` rather than ghost games. A blank there might be a ghost game or a recording gap, and we can't tell which. Leaving them out costs a few dozen matches, but it avoids treating missing data as an empty stadium.

In [13]:
in_ghost_window = df["date"].between("2020-03-01", "2021-07-31")
in_restriction = df["date"].between("2020-03-01", "2022-03-31")

# capacity proxy: the highest attendance a venue ever had outside the restriction period
venue_max = df[~in_restriction].groupby(["league", "Venue"])["Attendance"].max()
df["capacity_est"] = df.set_index(["league", "Venue"]).index.map(venue_max)
df["attendance_pct"] = (df["Attendance"] / df["capacity_est"]).clip(upper=1)

def crowd_status(row):
    if pd.isna(row["Attendance"]):
        return "none" if row["in_ghost_window"] else "unknown"
    # restricted crowd: under half the venue's normal maximum, or a temporary venue with no estimate
    if row["in_restriction"] and not (row["attendance_pct"] >= 0.5):
        return "partial"
    return "fans"

df["in_ghost_window"] = in_ghost_window
df["in_restriction"] = in_restriction
df["crowd_status"] = df.apply(crowd_status, axis=1)
df["attendance"] = df["Attendance"].fillna(0).where(df["crowd_status"] != "unknown")
# ghost games: 0% of the stadium was filled (not blank), so they show up in charts
df.loc[df["crowd_status"] == "none", "attendance_pct"] = 0.0
pd.crosstab([df["league"]], df["crowd_status"], margins=True)

crowd_status,fans,none,partial,unknown,All
league,,,,,
Bundesliga,2202,353,179,19,2753
EPL,2946,440,33,1,3420
La_Liga,2839,487,93,1,3420
Ligue_1,2591,376,110,19,3096
Serie_A,2684,466,264,6,3420
All,13262,2122,679,46,16109


In [14]:
check(df.loc[df["crowd_status"] == "none", "attendance"].eq(0).all(), "ghost games have attendance 0")
check(df.loc[df["crowd_status"] == "none", "attendance_pct"].eq(0).all(), "ghost games have attendance_pct 0")
check(df.loc[df["crowd_status"] == "unknown", "attendance"].isna().all(), "unknown matches have no attendance (excluded)")
ghost_outside = (df["crowd_status"] == "none") & ~df["in_ghost_window"]
check(not ghost_outside.any(), "no ghost games outside the ghost window")

full_in_window = (df["crowd_status"] == "fans") & df["in_restriction"] & (df["date"] >= "2020-04-01")
check(df.loc[full_in_window, "attendance_pct"].min() >= 0.5,
      f"every 'fans' match in the restriction period has at least 50% of normal capacity ({full_in_window.sum()} matches)")

no_cap = df["Attendance"].notna() & df["capacity_est"].isna()
print(f"\n{no_cap.sum()} matches with attendance but no capacity estimate (temporary venues):")
print(df.loc[no_cap, ["league", "season", "Home", "Venue", "Attendance"]].drop_duplicates("Venue").to_string())

check(df["Attendance"].max() < 100_000, f"no impossible attendance (max {df['Attendance'].max():,.0f})")

✅ ghost games have attendance 0
✅ ghost games have attendance_pct 0
✅ unknown matches have no attendance (excluded)
✅ no ghost games outside the ghost window
✅ every 'fans' match in the restriction period has at least 50% of normal capacity (815 matches)

2 matches with attendance but no capacity estimate (temporary venues):
        league  season    Home                Venue  Attendance
14159  Serie_A    2020  Spezia  Stadio Dino Manuzzi      1000.0
✅ no impossible attendance (max 97,939)


**About `capacity_est`:** we don't have official stadium capacities, so we use the highest attendance a venue had in a normal match. This is usually close to capacity. A team that only played at a venue during COVID has no estimate, so `attendance_pct` stays blank for it, but `attendance` (the raw number) is still there.

**Ghost games get `attendance_pct = 0`**, not blank, so they appear in charts of crowd size vs home advantage. Without this, the 2,122 most important matches would silently drop out of any plot.

**Temporary venues:** Spezia played its first 2020–21 home games at Cesena's stadium (Dino Manuzzi) while its own was being upgraded, so that venue has no normal-season maximum. Those matches count as `partial`, since every crowd in that period was restricted.

**Where `partial` vs `fans` gets decided:** inside the restriction window, a crowd under 50% of the venue's normal maximum counts as `partial`. This catches the 2020–21 trial crowds and the 2021–22 caps. Normal pre-lockdown matches in early March 2020 stay `fans`.

**Small crowds before COVID:** a few normal-season matches have very low attendance (one La Liga match shows just 13). These could be recording errors or stadium closures as punishment. They're outside our windows and only a handful, so we keep them as `fans`.

In [15]:
df.groupby("crowd_status")["attendance_pct"].describe().round(2)

,count,mean,std,min,25%,50%,75%,max
crowd_status,,,,,,,,
fans,13262.0,0.82,0.18,0.0,0.72,0.87,0.97,1.0
none,2122.0,0.00,0.00,0.0,0.00,0.00,0.00,0.0
partial,677.0,0.27,0.15,0.0,0.15,0.30,0.41,0.5
unknown,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 6. Build features for EDA

Everything is from the **home team's view**: a positive number favours the home team, or means more went against the away team.

| Feature | Meaning | Used for |
|---|---|---|
| `result` | H / D / A | Q1 home advantage |
| `home_points` | 3 / 1 / 0 | Q1 |
| `goal_diff` | home goals − away goals | Q1 |
| `foul_diff` | fouls called on away − on home | Q2 referee |
| `yellow_diff`, `red_diff` | cards to away − to home | Q2 referee |
| `shot_diff`, `sot_diff` | home shots − away shots (on target) | Q3 player performance |
| `crowd` | 1 = fans present (`fans` or `partial`), 0 = ghost game | all |

In [16]:
df["result"] = df["FTR"]
df["home_points"] = df["result"].map({"H": 3, "D": 1, "A": 0})
df["goal_diff"] = df["FTHG"] - df["FTAG"]
df["foul_diff"] = df["AF"] - df["HF"]
df["yellow_diff"] = df["AY"] - df["HY"]
df["red_diff"] = df["AR"] - df["HR"]
df["shot_diff"] = df["HS"] - df["AS"]
df["sot_diff"] = df["HST"] - df["AST"]
df["crowd"] = df["crowd_status"].map({"fans": 1, "partial": 1, "none": 0})

# data-entry error: one match lists more shots on target than shots -> blank the impossible values
bad_sot = (df["HST"] > df["HS"]) | (df["AST"] > df["AS"])
print("matches with shots on target > shots:")
print(df.loc[bad_sot, ["league", "date", "Home", "Away", "HS", "HST", "AS", "AST"]].to_string())
df.loc[df["HST"] > df["HS"], ["HST"]] = np.nan
df.loc[df["AST"] > df["AS"], ["AST"]] = np.nan
df["sot_diff"] = df["HST"] - df["AST"]

matches with shots on target > shots:
     league       date       Home      Away    HS  HST   AS  AST
4538    EPL 2021-08-15  Newcastle  West Ham  17.0  3.0  8.0  9.0


In [17]:
check(set(df["result"]) == {"H", "D", "A"}, "result only has H / D / A")
check(((df["result"] == "H") == (df["goal_diff"] > 0)).all(), "result matches goal difference")
stat_cols = ["FTHG", "FTAG", "HS", "AS", "HST", "AST", "HF", "AF", "HY", "AY", "HR", "AR"]
missing = int(df[stat_cols].isna().sum().sum())
check(missing <= 1, f"match stats complete ({missing} value blanked as a data error)")
check((df[stat_cols].fillna(0) >= 0).all().all(), "no negative stats")
check(not ((df["HST"] > df["HS"]) | (df["AST"] > df["AS"])).any(), "shots on target never exceed shots")
check(df["HR"].between(0, 5).all() and df["AR"].between(0, 5).all(), "red cards in a sensible range")

✅ result only has H / D / A
✅ result matches goal difference
✅ match stats complete (1 value blanked as a data error)
✅ no negative stats
✅ shots on target never exceed shots
✅ red cards in a sensible range


## 7. Save

Keep only the columns EDA needs, with clear names.

In [18]:
final = df.rename(columns={
    "FTHG": "home_goals", "FTAG": "away_goals",
    "HS": "home_shots", "AS": "away_shots", "HST": "home_sot", "AST": "away_sot",
    "HF": "home_fouls", "AF": "away_fouls", "HY": "home_yellows", "AY": "away_yellows",
    "HR": "home_reds", "AR": "away_reds", "Home": "home_team", "Away": "away_team", "Venue": "venue",
})[[
    "league", "season", "date", "home_team", "away_team", "venue", "referee",
    "attendance", "capacity_est", "attendance_pct", "crowd_status", "crowd",
    "home_goals", "away_goals", "result", "home_points", "goal_diff",
    "home_shots", "away_shots", "home_sot", "away_sot", "shot_diff", "sot_diff",
    "home_fouls", "away_fouls", "home_yellows", "away_yellows", "home_reds", "away_reds",
    "foul_diff", "yellow_diff", "red_diff", "awarded",
]].sort_values(["league", "date", "home_team"]).reset_index(drop=True)

final.to_csv(CLEAN / "matches.csv", index=False)
print(final.shape)
final.head()

(16109, 33)


,league,season,date,home_team,away_team,venue,referee,attendance,capacity_est,attendance_pct,...,home_fouls,away_fouls,home_yellows,away_yellows,home_reds,away_reds,foul_diff,yellow_diff,red_diff,awarded
0,Bundesliga,2017,2017-08-18,Bayern Munich,Leverkusen,Allianz Arena,Tobias Stieler,75000.0,75000.0,1.000000,...,13.0,17.0,1.0,2.0,0.0,0.0,4.0,1.0,0.0,False
1,Bundesliga,2017,2017-08-19,Hamburger SV,Augsburg,Volksparkstadion,Daniel Siebert,49449.0,57000.0,0.867526,...,18.0,16.0,4.0,3.0,0.0,0.0,-2.0,-1.0,0.0,False
2,Bundesliga,2017,2017-08-19,Hertha BSC,Stuttgart,Olympiastadion Berlin,Sascha Stegemann,44751.0,74669.0,0.599325,...,18.0,12.0,2.0,1.0,0.0,0.0,-6.0,-1.0,0.0,False
3,Bundesliga,2017,2017-08-19,Hoffenheim,Werder Bremen,Dietmar-Hopp-Stadion,Guido Winkmann,30150.0,30150.0,1.000000,...,17.0,15.0,1.0,3.0,0.0,0.0,-2.0,2.0,0.0,False
4,Bundesliga,2017,2017-08-19,Mainz 05,Hannover 96,Opel Arena,Felix Brych,28279.0,34000.0,0.831735,...,18.0,14.0,3.0,5.0,0.0,0.0,-4.0,2.0,0.0,False


## Final check

A first look at whether the crowd matters. This isn't the analysis yet (that's `03_eda`), just a check that the numbers make sense.

In [19]:
summary = final.dropna(subset=["crowd"]).groupby("crowd_status").agg(
    matches=("result", "size"),
    home_win_pct=("result", lambda r: (r == "H").mean() * 100),
    away_win_pct=("result", lambda r: (r == "A").mean() * 100),
    avg_home_points=("home_points", "mean"),
    avg_goal_diff=("goal_diff", "mean"),
    avg_yellow_diff=("yellow_diff", "mean"),
    avg_foul_diff=("foul_diff", "mean"),
).round(2)
summary

,matches,home_win_pct,away_win_pct,avg_home_points,avg_goal_diff,avg_yellow_diff,avg_foul_diff
crowd_status,,,,,,,
fans,13262,44.10,30.68,1.58,0.30,0.27,0.32
none,2122,39.87,34.83,1.45,0.14,-0.01,-0.23
partial,679,44.18,30.49,1.58,0.38,0.27,0.35


In [20]:
check(len(final) == len(fd) - 2, f"no matches lost in cleaning except the 2 incomplete ones ({len(final)} rows)")
check(final[["league", "season", "home_team", "away_team"]].duplicated().sum() == 0, "every match appears once")
print("\ncrowd status:", final["crowd_status"].value_counts().to_dict())

✅ no matches lost in cleaning except the 2 incomplete ones (16109 rows)
✅ every match appears once

crowd status: {'fans': 13262, 'none': 2122, 'partial': 679, 'unknown': 46}


## Cleaning decisions (for your report)

1. **Removed:** 101 cancelled Ligue 1 2019–20 fixtures and 21 relegation play-off matches.
2. **Team names:** mapped 39 DataHub names to FBref spellings, and checked that every team matched.
3. **Score source:** DataHub's on-pitch result. Two matches were officially "awarded" afterwards and are flagged with `awarded`.
4. **Referee:** taken from FBref, because DataHub only has EPL referees.
5. **Ghost games:** blank attendance from 1 Mar 2020 to 31 Jul 2021 = 0 fans. Blank attendance outside that window = `unknown`, excluded from crowd analysis.
6. **Partial crowds:** matches from 1 Mar 2020 to 31 Mar 2022 with under 50% of the venue's normal maximum, or played at a temporary venue.
7. **Capacity:** estimated as each venue's highest attendance outside the restriction period.
9. **Data error:** one EPL match (Aug 2021) listed more shots on target than shots, so that value was blanked.
8. **Limitation:** some 2021–22 blanks are real ghost games (Bundesliga, Omicron wave), but they're marked `unknown` because they can't be told apart from recording gaps.